### Setup
Autoreload and imports

In [1]:
%load_ext autoreload
%autoreload 2

In [18]:
# Imports
import sys
from pathlib import Path
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, SaliencyMapMethod, CarliniL2Method

import art.attacks.evasion.projected_gradient_descent.projected_gradient_descent_pytorch as _pgd_pt
_pgd_pt.compute_success = lambda *a, **kw: 0.0

sys.path.append(str(Path.cwd().parents[2]))

from utils.functions import get_windowed_data
from utils.notebook import get_model_classifier, clean_data_test, adv_test, FilenameLoader

### Define Inputs

In [4]:
## Inputs
checkpoint_name, data_name, save_name = FilenameLoader.const_pos()

checkpoint_file= f"../../../saved_models/{checkpoint_name}"
data_file = f"../../../data/{data_name}"
save_path = f"../final_data/{save_name}"

collapsed = False # True for JSMA and CW
save_clean = False

### Load Model and Data
Load the data and model and run on clean data to check the accuracy/F1.

In [5]:
## Load model and data
# Load original model and ART-wrapper classifier
model, classifier = get_model_classifier(checkpoint_file, collapsed)

# Load data
(x_train, y_train), (x_test, y_test), fed_dataset, scaler = get_windowed_data(data_file, 
                                                                      normalize=True, 
                                                                      train_perc=80)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/anaconda3/envs/reu/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!


In [6]:
## Run clean data test
out = clean_data_test(model = model, classifier = classifier, # model information
                x_test=x_test, y_test=y_test, # data information
                checkpoint_file=checkpoint_file, data_file=data_file, # to save in json
                save_path=save_path, filename="clean.json", # saving information
                save_results=save_clean,
                collapsed=collapsed
                ) 

# Check if output passes
print('noWrapper:', 'PASS' if out['noWrapper']['f1'] > 0.8 else 'FAIL')
print('wrapper:', 'PASS' if out['wrapper']['f1'] > 0.8 else 'FAIL')

save=False, Metrics not saved
Metrics {'noWrapper': {'accuracy': 0.8897366438520846, 'recall': 0.9759886700836256, 'precision': 0.7376457251905284, 'f1': 0.840242087001758, 'falseNegativeRate': 0.024011329916374426, 'falsePositiveRate': 0.14671964790659492, 'TP': 361799, 'TN': 748361, 'FP': 128679, 'FN': 8901}, 'wrapper': {'accuracy': 0.8897366438520846, 'precision': 0.7376457251905284, 'recall': 0.9759886700836256, 'f1': 0.840242087001758, 'falseNegativeRate': 0.024011329916374426, 'falsePositiveRate': 0.14671964790659492, 'TP': 361799, 'TN': 748361, 'FP': 128679, 'FN': 8901}, 'files': {'checkpointFile': '../../../saved_models/ConstantPos-final.ckpt', 'dataFile': '../../../data/ConstPos_0709.csv'}}
noWrapper: PASS
wrapper: PASS


### Adversarial Test
Run adversarial attacks and save the outputted metrics

In [ ]:
# running: constpos
print(save_path)

for i in range(1, 51):
    eps = float(i/100)
    adv_test(
        classifier = classifier, # classifier
        x_test=x_test, y_test=y_test, # test data
        checkpoint_file=checkpoint_file, data_file=data_file, # for json
        
        end_index = len(y_test.numpy()),
        path = f"{save_path}/pgd-10-2",
        filename=f"adv_eps_{eps}.json",
        collapsed=collapsed,
        Attack=ProjectedGradientDescent,
        eps=eps, 
        targeted=True,
        max_iter = 10
        )

../final_data/constpos
=== Attack: ProjectedGradientDescent, kwargs: {'eps': 0.01, 'targeted': True, 'max_iter': 10} ===


Accuracy:           0.5351
Precision:          0.3477
Recall:             0.6448
F1:                 0.4518
ASR (FNR):          0.3552
False Positive Rate:0.5113
TP=239035, TN=428570, FP=448470, FN=131665
Time elapsed:       213.69s
Saved metrics to ../final_data/constpos/pgd-10-2/adv_eps_0.01.json
=== Attack: ProjectedGradientDescent, kwargs: {'eps': 0.02, 'targeted': True, 'max_iter': 10} ===


Accuracy:           0.3336
Precision:          0.1849
Recall:             0.3646
F1:                 0.2454
ASR (FNR):          0.6354
False Positive Rate:0.6794
TP=135167, TN=281136, FP=595904, FN=235533
Time elapsed:       281.87s
Saved metrics to ../final_data/constpos/pgd-10-2/adv_eps_0.02.json
=== Attack: ProjectedGradientDescent, kwargs: {'eps': 0.03, 'targeted': True, 'max_iter': 10} ===


Accuracy:           0.2759
Precision:          0.0890
Recall:             0.1556
F1:                 0.1132
ASR (FNR):          0.8444
False Positive Rate:0.6733
TP=57664, TN=286539, FP=590501, FN=313036
Time elapsed:       280.60s
Saved metrics to ../final_data/constpos/pgd-10-2/adv_eps_0.03.json
=== Attack: ProjectedGradientDescent, kwargs: {'eps': 0.04, 'targeted': True, 'max_iter': 10} ===


Accuracy:           0.2674
Precision:          0.0559
Recall:             0.0923
F1:                 0.0697
ASR (FNR):          0.9077
False Positive Rate:0.6586
TP=34231, TN=299427, FP=577613, FN=336469
Time elapsed:       285.32s
Saved metrics to ../final_data/constpos/pgd-10-2/adv_eps_0.04.json
=== Attack: ProjectedGradientDescent, kwargs: {'eps': 0.05, 'targeted': True, 'max_iter': 10} ===


...

Accuracy:           0.2591
Precision:          0.0523
Recall:             0.0872
F1:                 0.0653
ASR (FNR):          0.9128
False Positive Rate:0.6682
TP=32308, TN=291035, FP=586005, FN=338392
Time elapsed:       328.43s
Saved metrics to ../final_data/constpos/pgd-10-2/adv_eps_0.12.json
=== Attack: ProjectedGradientDescent, kwargs: {'eps': 0.13, 'targeted': True, 'max_iter': 10} ===


Accuracy:           0.2378
Precision:          0.0252
Recall:             0.0415
F1:                 0.0313
ASR (FNR):          0.9585
False Positive Rate:0.6792
TP=15368, TN=281363, FP=595677, FN=355332
Time elapsed:       307.68s
Saved metrics to ../final_data/constpos/pgd-10-2/adv_eps_0.13.json
=== Attack: ProjectedGradientDescent, kwargs: {'eps': 0.14, 'targeted': True, 'max_iter': 10} ===


Accuracy:           0.2241
Precision:          0.0115
Recall:             0.0189
F1:                 0.0143
ASR (FNR):          0.9811
False Positive Rate:0.6891
TP=7012, TN=272655, FP=604385, FN=363688
Time elapsed:       289.01s
Saved metrics to ../final_data/constpos/pgd-10-2/adv_eps_0.14.json
=== Attack: ProjectedGradientDescent, kwargs: {'eps': 0.15, 'targeted': True, 'max_iter': 10} ===


Accuracy:           0.2154
Precision:          0.0068
Recall:             0.0113
F1:                 0.0085
ASR (FNR):          0.9887
False Positive Rate:0.6983
TP=4181, TN=264638, FP=612402, FN=366519
Time elapsed:       256.78s
Saved metrics to ../final_data/constpos/pgd-10-2/adv_eps_0.15.json
=== Attack: ProjectedGradientDescent, kwargs: {'eps': 0.16, 'targeted': True, 'max_iter': 10} ===


PGD - Batches:  42%|████▏     | 1636/3900 [01:47<02:32, 14.84it/s]